In [1]:
import os
import pathlib
import pandas as pd
from rapidfuzz import fuzz

In [2]:
root_path = pathlib.Path(os.getcwd())
root_path = root_path.parents[1]

# Nettoyage

In [ ]:
dataframe_product_detail = pd.read_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export.xlsx",
    )
)
dataframe_product_detail

In [ ]:
dataframe_product_detail.dtypes

In [ ]:
dataframe_product_detail = dataframe_product_detail.rename(
    columns={
        "DISTRIBUTEUR": "distributor",
        "SOURCE DONNEES": "data_source",
        "CODE PRODUIT": "product_code",
        "DESCRIPTION": "description",
        "MARQUE": "brand",
        "INDUSTRIEL": "industrial",
        "UF": "unit",
        "Qté Facture": "quantity",
        "Montant HT": "amount_ht"
    }
)
dataframe_product_detail

In [ ]:
dataframe_product_detail_distributor = dataframe_product_detail["distributor"].isna().sum()
dataframe_product_detail_distributor

In [ ]:
dataframe_product_detail.dropna(how="all", inplace=True)
dataframe_product_detail

In [ ]:
dataframe_product_detail.drop(index=719, inplace=True) # À modifier si jamais la ligne 719 n'est plus vide et prendre la dernière ligne
dataframe_product_detail

In [ ]:
# Add new column after data_source column
dataframe_product_detail.insert(
    dataframe_product_detail.columns.get_loc("data_source") + 1,
    "product_name",
    "Product Name"
)
dataframe_product_detail

In [ ]:
dataframe_product_detail["amount_ht"] = dataframe_product_detail["amount_ht"].round(2)
dataframe_product_detail

In [ ]:
dataframe_product_detail["unit"] = dataframe_product_detail["unit"].str.upper()

dictionary_unit = {
    # BOCAL
    "BCL": "BOCAL",
    # BIDON
    "BID": "BIDON",
    # BOUTEILLE
    "BLLE": "BOUTEILLE",
    # BOÎTE
    "BT": "BOÎTE",
    "BT.": "BOÎTE",
    "BTE": "BOÎTE",
    "BOITE": "BOÎTE",
    # BRIQUE
    "BRQ": "BRIQUE",
    # COFFRET
    "CO": "COFFRET",
    "COF": "COFFRET",
    # COLIS
    "COL": "COLIS",
    # FLACON
    "FLC": "FLACON",
    # PIÈCE
    "PI": "PIÈCE",
    # POCHE
    "PCH": "POCHE",
    # SEAU
    "SEA": "SEAU",
    # UNITÉ
    "U": "UNITÉ"
}

dataframe_product_detail["unit"] = dataframe_product_detail["unit"].replace(dictionary_unit)

In [ ]:
dataframe_product_detail["product_code"] = dataframe_product_detail["product_code"].str.upper()

dictionary_brand = {
    # HELLMANN'S
    "HELLEMANSQUEEZE": "HELLMANN'S SQUEEZE",
    "HELLMANNSQUEEZE": "HELLMANN'S SQUEEZE",
    # AMORA
    "SAVORA": "AMORASAVORA"
}

dataframe_product_detail["product_code"] = dataframe_product_detail["product_code"].replace(dictionary_brand)

description_contains_brand = ["AMORA", "HELLMANN'S", "KNORR", "MAILLE", "MAIZENA", "TABASCO", "LIPTON", "ELEPHANT"]

for brand in description_contains_brand:
    dataframe_product_detail.loc[
        dataframe_product_detail["description"].str.contains(brand, case=False, na=False) |
        dataframe_product_detail["product_code"].str.contains(brand, case=False, na=False),
        "brand"
    ] = brand

In [ ]:
for column in dataframe_product_detail.select_dtypes(include=["object"]):
    dataframe_product_detail[column] = dataframe_product_detail[column].str.title().str.replace(r"(?<=')([A-Z])", lambda value: value.group(0).lower(), regex=True)

In [ ]:
dataframe_product_detail

In [ ]:
dataframe_product_detail.to_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export_cleaned.xlsx",
    ),
    index=False
)

# Score de similarité

In [10]:
product_file = pd.read_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export_cleaned.xlsx",
    )
)
mapping_file = pd.read_excel(
    os.path.join(
        root_path,
        "data_folder",
        "mapping",
        "mapping_product.xlsx",
    ),
    sheet_name="mapping_products"
)

In [20]:
def find_product_name(param_row):
    """ Find the product name based on the description and brand of the product """

    description_product = str(param_row["description"]).upper()
    brand_product = str(param_row["brand"]).upper()

    candidates = mapping_file[mapping_file["brand"].str.upper() == brand_product]

    best_score = 0
    best_product_name = None
    for _, rule in candidates.iterrows():
        category = str(rule["categories"]).upper()

        score = fuzz.token_set_ratio(description_product, category)

        product_file["matching_score"] = score
        if score > best_score:
            best_score = score
            best_product_name = rule["product_name"]


    if best_score >= 70:
        return best_product_name

    return None

product_file["product_name"] = product_file.apply(find_product_name, axis=1)

In [21]:
product_file.to_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export_final.xlsx",
    ),
    index=False
)